A small retail chain with 3 stores has given you their raw sales data in a spreadsheet. They want answers to specific business questions and a clean summary they can share with their investors. 

In [0]:
from pyspark.sql.types import (
    StructType, StructField,
    IntegerType, StringType, DoubleType
)
from pyspark.sql.functions import col, round, when, sum, avg, count

schema = StructType([
    StructField("sale_id",     IntegerType(), True),
    StructField("store",       StringType(),  True),
    StructField("rep",         StringType(),  True),
    StructField("category",    StringType(),  True),
    StructField("product",     StringType(),  True),
    StructField("amount",      DoubleType(),  True),
    StructField("quantity",    IntegerType(), True),
    StructField("sale_date",   StringType(),  True),
    StructField("customer_id", IntegerType(), True),
])

data = [
    (1,  "Store A", "Alice", "Electronics", "Laptop",  1200.0, 1, "2024-01-05", 101),
    (2,  "Store B", "Bob",   "Clothing",    "Jacket",    89.99, 2, "2024-01-05", 102),
    (3,  "Store A", "Alice", "Electronics", "Phone",    750.0, 1, "2024-01-06", 103),
    (4,  "Store C", "Carol", "Food",        "Hamper",    45.0, 3, "2024-01-06", 101),
    (5,  "Store B", "Bob",   "Electronics", "Laptop",  1100.0, 1, "2024-01-07", 104),
    (6,  "Store A", "Diana", "Clothing",    "Shoes",    199.0, 2, "2024-01-07", 102),
    (7,  "Store C", "Carol", "Electronics", "Tablet",   600.0, 1, "2024-01-08", 105),
    (8,  "Store B", "Bob",   "Food",        "Hamper",    55.0, 4, "2024-01-08", 103),
    (9,  "Store A", "Alice", "Electronics", "Laptop",   950.0, 1, "2024-01-09", 106),
    (10, "Store C", "Carol", "Clothing",    "Jacket",   120.0, 1, "2024-01-09", 104),
    (11, "Store A", "Diana", "Food",        "Hamper",    38.0, 2, "2024-01-10", 107),
    (12, "Store B", "Bob",   "Electronics", "Phone",    780.0, 1, "2024-01-10", 105),
    (13, "Store C", "Carol", "Electronics", "Laptop",  1300.0, 1, "2024-01-11", 101),
    (14, "Store A", "Alice", "Clothing",    "Shoes",    220.0, 1, "2024-01-11", 108),
    (15, "Store B", "Bob",   "Clothing",    "Jacket",    95.0, 3, "2024-01-12", 109),
    (16, "Store C", "Carol", "Food",        "Hamper",    60.0, 2, "2024-01-12", 102),
    (17, "Store A", "Diana", "Electronics", "Tablet",   580.0, 1, "2024-01-13", 103),
    (18, "Store B", "Bob",   "Electronics", "Laptop",  1050.0, 1, "2024-01-13", 110),
    (19, "Store A", "Alice", "Food",        "Hamper",    42.0, 1, "2024-01-14", 104),
    (20, "Store C", "Carol", "Electronics", "Phone",    710.0, 1, "2024-01-14", 106),
]

df = spark.createDataFrame(data, schema)
df.show()
print("Row count:", df.count())

In [0]:
df.write\
    .format("delta")\
    .mode("overwrite")\
    .saveAsTable("retails_bronze")

In [0]:
spark.read.table("retails_bronze").show()

In [0]:
spark.sql("DROP TABLE IF EXISTS retail_silver")
df_silver =(
    spark.read.table("retails_bronze")
    .withColumn("total_value", col("amount")* col("quantity"))
    .withColumn("order_size", when(col("amount") >= 500 , "large").otherwise("small"))
    .filter (col("amount") >= 30)
    .select("sale_id", "store", "rep", "category", "product",
            "amount", "quantity", "total_value", "order_size",
            "sale_date", "customer_id")
)
df_silver.show()
df_silver.write\
    .format("delta")\
    .mode("overwrite")\
    .saveAsTable("retail_silver")
spark.read.table("retail_silver").show()

In [0]:
df_gold1 = (
    spark.read.table("retail_silver")
    .groupBy("store")
    .agg(
        sum("amount").alias("total_revenue"),
        count("sale_id").alias("total_orders"),
        round(avg("amount"),2).alias("avg_order_value")
    )
    .sort(col("total_revenue").desc())
)

df_gold1.show()
df_gold1.write\
    .format("delta")\
    .mode("overwrite")\
    .saveAsTable("retail_gold_by_store")


In [0]:
df_gold2 = (
    spark.read.table("retail_silver")
    .groupBy("rep","store")
    .agg(
        sum("amount").alias("total_revenue"),
        count("sale_id").alias("total_orders"),
        round(avg("amount"),2).alias("avg_order_value")
    )
    .sort(col("total_revenue").desc())
)

df_gold2.show()
df_gold2.write\
    .format("delta")\
    .mode("overwrite")\
    .saveAsTable("retail_gold_by_rep")

In [0]:
%sql
select total_revenue, store from retail_gold_by_store
order by total_revenue desc
limit 1

In [0]:
%sql
select total_orders, rep from retail_gold_by_rep
order by total_orders desc
limit 1

In [0]:
%sql
select category, sum(amount) as total_revenue from retail_silver
group by category
order by total_revenue desc
limit 1
    

In [0]:
%sql
select customer_id, sum(amount) as total_spent from retail_silver
group by customer_id
order by total_spent desc
limit 1

In [0]:
%sql
WITH rep_ranked AS (
    SELECT
        store,
        rep,
        ROUND(SUM(amount), 2) AS total_revenue,
        RANK() OVER (PARTITION BY store ORDER BY SUM(amount) DESC) AS store_rank
    FROM retail_silver
    GROUP BY store, rep
)
SELECT store, rep, total_revenue
FROM rep_ranked
WHERE store_rank = 1
ORDER BY store


In [0]:
%sql
select 
    category,
    round(sum(amount) / sum(sum(amount)) over() * 100, 1) as pct_of_total
from retail_silver
group by category


In [0]:
%sql
WITH daily_totals AS (
    SELECT
        sale_date,
        ROUND(SUM(amount), 2) AS daily_revenue
    FROM retail_silver
    GROUP BY sale_date
)
SELECT
    sale_date,
    daily_revenue,
    ROUND(SUM(daily_revenue) OVER (
        ORDER BY sale_date
        ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
    ), 2) AS running_total
FROM daily_totals
ORDER BY sale_date

In [0]:
%sql
with store_metrics as (
    select
        store,
        round(sum(amount), 2) as total_revenue,
        count(sale_id) as total_orders,
        round(avg(amount), 2) as avg_order_value
    from retail_silver
    group by store
),
store_top_category as(
    select
        store,
        category,
        round(sum(amount),2) as total_revenue,
        rank() over(partition by store order by sum(amount) desc) as category_rank
    from retail_silver
    group by store, category
),
store_top_rep as(
    select
        store,
        rep,
        round(sum(amount),2) as total_revenue,
        rank() over(partition by store order by sum(amount) desc) as rep_rank
    from retail_silver
    group by store, rep

),
final as(
    select
        m.store,
        m.total_revenue,
        m.total_orders,
        m.avg_order_value,
        tc.category as top_category,
        tr.rep as top_rep,
        rank() over (order by m.total_revenue desc) as revenue_rank,
        round(m.total_revenue/sum(m.total_revenue) over() *100, 1)
    from store_metrics m
    join store_top_category tc on m.store = tc.store and tc.category_rank =1
    join store_top_rep tr on m.store = tr.store and tr.rep_rank = 1
)
select * from final
order by revenue_rank
